# OCT-OLMo across eleven constitutions

`python scripts/run.py runs/oct_olmo/<trait>/spec.py`, eleven times. Same 15 models and
same 200 scenarios in every run, varying only the constitution.

~9,000 generations per run. Each run checkpoints next to its spec, so re-running the last
cell resumes rather than restarts — that is the normal way to recover from a dead pod.

In [ ]:
import os

# Keep the HF cache off /workspace: on RunPod that is MooseFS, which this
# pipeline's checkpoint already had to be rewritten to survive.
for var in ("HF_HOME", "HUGGINGFACE_HUB_CACHE", "TRANSFORMERS_CACHE"):
    os.environ[var] = "/root/.cache/huggingface"

In [ ]:
!git clone https://github.com/jchang153/EigenBench.git
%cd EigenBench
!git checkout olmo-runs
!pip install -q -r requirements.txt

In [ ]:
import getpass
import os

from huggingface_hub import notebook_login

notebook_login()  # the OLMo personas are private adapters
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ").strip()
os.environ["SPACE_SECRET"] = getpass.getpass("SPACE_SECRET: ").strip()

Check the plan before spending GPU time — `--estimate-calls` runs no model.

In [ ]:
!python scripts/run.py runs/oct_olmo/goodness/spec.py --estimate-calls

Sequential, because they share one GPU. A failed run does not stop the batch; re-run this
cell to resume whatever did not finish.

In [ ]:
TRAITS = [
    "goodness", "humor", "impulsiveness", "loving", "mathematical",
    "misalignment", "nonchalance", "poeticism", "remorse", "sarcasm",
    "sycophancy", "kindness"
]

for trait in TRAITS:
    print(f"\n{'=' * 60}\n  {trait}\n{'=' * 60}")
    !python scripts/run.py runs/oct_olmo/{trait}/spec.py 2>&1 | tee runs/oct_olmo/{trait}/run.log

Notes:

- Every one of these specs re-collects from scratch. The generation budgets and the model
  roster are both fingerprinted and both changed, so an older checkpoint is refused rather
  than silently reused.
- Each run starts its own vLLM engine over the same base and adapters, so the batch pays
  engine startup eleven times.
- `kindness` is not in this set: different constitution, 8 criteria, no per-model budgets.